# 3. Data Preparation

## 3.1 Objectives
Based on the EDA, we identified two main issues:
1. **Encoding**: The attack labels contain special characters (e.g., `Web Attack � Brute Force`) due to non-UTF8 encoding in the raw CSV. We will fix this using `latin-1`.
2. **Data Quality**: `Flow Bytes/s` and `Flow Packets/s` contain `inf` and `NaN` values.

## 3.2 Strategy
1. Load the raw data with the correct encoding.
2. Replace `inf` with `NaN`.
3. Drop rows with `NaN` (since they represent a tiny fraction of the data).
4. Create a binary target: `0` for BENIGN, `1` for any Attack.
5. Perform a stratified train/test split to maintain the class imbalance.

In [3]:
import pandas as pd
import numpy as np
import glob
import os
from sklearn.model_selection import train_test_split

# 1. Charger le fichier avec le bon encodage
csv_files = glob.glob("../data/raw/**/*.csv", recursive=True)
csv_files = [f for f in csv_files if "WorkingHours" in f]
file_path = csv_files[0]

df = pd.read_csv(file_path, nrows=100000, encoding='latin-1')

# Nettoyage des colonnes
df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.duplicated()]

# 2. CORRECTION ENCODAGE : On supprime tous les caractères non-ASCII (les fameux ï¿½)
df['Label'] = df['Label'].str.replace(r'[^\x00-\x7F]+', ' ', regex=True).str.strip()

print(f"Avant nettoyage : {df.shape[0]} lignes")

# 3. Remplacer les infinis par NaN
df = df.replace([np.inf, -np.inf], np.nan)

# 4. CORRECTION WARNING : On n'utilise plus inplace=True
df = df.dropna()
print(f"Après nettoyage (drop NaN) : {df.shape[0]} lignes")

# 5. Vérifier que les caractères bizarres ont disparu
print("\nClasses d'attaques nettoyées :")
print(df['Label'].unique())

Avant nettoyage : 100000 lignes
Après nettoyage (drop NaN) : 99910 lignes

Classes d'attaques nettoyées :
<ArrowStringArray>
[                    'BENIGN',   'Web Attack   Brute Force',
           'Web Attack   XSS', 'Web Attack   Sql Injection']
Length: 4, dtype: str


In [4]:
# 1. Créer la cible binaire (0 = Benign, 1 = Attack)
df['Target'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print("Distribution de la nouvelle cible :")
print(df['Target'].value_counts(normalize=True) * 100)

# 2. Séparer les features (X) et la cible (y)
X = df.drop(columns=['Label', 'Target'])
y = df['Target']

# 3. Split stratifié (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTaille du Train set : {X_train.shape[0]} lignes")
print(f"Taille du Test set : {X_test.shape[0]} lignes")
print(f"Ratio d'attaques dans le Train : {y_train.mean()*100:.2f}%")
print(f"Ratio d'attaques dans le Test : {y_test.mean()*100:.2f}%")

# 4. SAUVEGARDE EN PARQUET (Pour le prochain notebook)
os.makedirs("../data/processed", exist_ok=True)

X_train.to_parquet("../data/processed/X_train.parquet")
X_test.to_parquet("../data/processed/X_test.parquet")
y_train.to_frame().to_parquet("../data/processed/y_train.parquet")
y_test.to_frame().to_parquet("../data/processed/y_test.parquet")

print("\n✅ Données d'entraînement et de test sauvegardées dans data/processed/ !")

Distribution de la nouvelle cible :
Target
0    97.818036
1     2.181964
Name: proportion, dtype: float64

Taille du Train set : 79928 lignes
Taille du Test set : 19982 lignes
Ratio d'attaques dans le Train : 2.18%
Ratio d'attaques dans le Test : 2.18%

✅ Données d'entraînement et de test sauvegardées dans data/processed/ !


## 3.3 Conclusion

The data is now clean:
- Infinite values are removed.
- The target is binary (BENIGN vs ATTACK).
- The train/test split is stratified, ensuring both sets have the same proportion of attacks.

**Next Step:** We will train our Machine Learning models (XGBoost and Isolation Forest) in `04_modeling.ipynb`.